# Gated Fusion Experiment: RGB + Depth + Joint Encoders

## Objective
This notebook implements a **multimodal gated fusion model** for posture classification using three pretrained encoders:

- **RGB encoder**
- **Depth encoder**
- **Joint encoder**

The goal is to combine feature representations from all three modalities and train a **gated fusion classification head** on top of them.

---

## Experiment Setup
In this experiment:

- pretrained encoder weights are loaded from saved artifacts
- encoder backbones are used as **feature extractors**
- encoder parameters are initially **frozen**
- a **gated fusion module** is trained to combine modality-specific features
- the final classifier predicts one of the posture classes:
  - **supine**
  - **left**
  - **right**

---

## Why Gated Fusion
Gated fusion allows the model to learn the **relative importance of each modality** for a given sample.

This is useful because:

- **RGB** may become less reliable under blanket occlusion
- **Depth** may retain useful spatial posture information
- **Joint features** may provide structural cues but may also be noisy

The gating mechanism helps the model dynamically weight each modality before final classification.

## Imports and Setup

In [57]:
# =========================
# Core
# =========================
from pathlib import Path

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

In [58]:
# =========================
# Device setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Paths and Configuration

In [59]:
# =========================
# Paths
# =========================
PROJECT_ROOT = Path.cwd().resolve().parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

DEPTH_ARTIFACT_DIR = ARTIFACTS_DIR / "depth_encoder_cnn" / "checkpoints"
RGB_ARTIFACT_DIR = ARTIFACTS_DIR / "rgb_encoder_cnn" / "checkpoints"
JOINT_ARTIFACT_DIR = ARTIFACTS_DIR / "joint_xyo" / "checkpoints"

FUSION_CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints_fusion_gated"
FUSION_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Depth artifacts:", DEPTH_ARTIFACT_DIR)
print("RGB artifacts:", RGB_ARTIFACT_DIR)
print("Joint artifacts:", JOINT_ARTIFACT_DIR)
print("Fusion checkpoints:", FUSION_CHECKPOINT_DIR)

Project root: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense
Depth artifacts: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\depth_encoder_cnn\checkpoints
RGB artifacts: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints
Joint artifacts: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\joint_xyo\checkpoints
Fusion checkpoints: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\checkpoints_fusion_gated


## Experiment Configuration

Define the basic experiment settings for gated fusion, including:

- number of classes
- batch size
- learning rate
- number of epochs
- common feature dimension for fusion

These can be adjusted later if needed.

In [60]:
# =========================
# Experiment config
# =========================
NUM_CLASSES = 3

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

COMMON_FEATURE_DIM = 256 # to convert all features from encoders to a common size
DROPOUT = 0.3

CLASS_NAMES = ["supine", "left", "right"]

## Encoder Checkpoint Paths

In [61]:
# =========================
# Encoder checkpoint files
# =========================
DEPTH_CHECKPOINT_PATH = DEPTH_ARTIFACT_DIR / "best_depth_encoder_only.pth"
RGB_CHECKPOINT_PATH = RGB_ARTIFACT_DIR / "best_rgb_encoder.pt"
JOINT_CHECKPOINT_PATH = JOINT_ARTIFACT_DIR / "joint_encoder_xyo_RGB.pth"

print("Depth checkpoint:", DEPTH_CHECKPOINT_PATH)
print("RGB checkpoint:", RGB_CHECKPOINT_PATH)
print("Joint checkpoint:", JOINT_CHECKPOINT_PATH)

Depth checkpoint: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\depth_encoder_cnn\checkpoints\best_depth_encoder_only.pth
RGB checkpoint: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints\best_rgb_encoder.pt
Joint checkpoint: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\joint_xyo\checkpoints\joint_encoder_xyo_RGB.pth


In [62]:
# =========================
# Dataset and metadata
# =========================

import pandas as pd
import numpy as np
import scipy.io as sio
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DATASET_ROOT = PROJECT_ROOT.parent / "SLP2022" / "SLP" / "danaLab"
CSV_PATH = DATASET_ROOT / "posture_labels_all_modalities.csv"

def build_fusion_metadata(csv_path):
    df = pd.read_csv(csv_path)
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # use RGB rows as the aligned base sample definition
    rgb_df = df[df["modality"] == "RGB"].copy()

    rgb_df["rgb_path"] = (
        rgb_df["subject_id"] + "/RGB/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )

    rgb_df["depth_path"] = (
        rgb_df["subject_id"] + "/depth/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )

    # joints shared at subject level
    rgb_df["joint_file"] = rgb_df["subject_id"] + "/joints_gt_RGB.mat"
    rgb_df["frame_idx_0based"] = rgb_df["image_index"] - 1

    return rgb_df[[
        "subject_id",
        "condition",
        "image_index",
        "label",
        "label_id",
        "rgb_path",
        "depth_path",
        "joint_file",
        "frame_idx_0based",
    ]].reset_index(drop=True)

# =========================
# Fusion dataset and data loading
# =========================

IMG_W = 576.0
IMG_H = 1024.0

def preprocess_joint_frame_xyo(frame_joints):
    x = frame_joints[0].astype(np.float32) / IMG_W
    y = frame_joints[1].astype(np.float32) / IMG_H
    occ = frame_joints[2].astype(np.float32)
    out = np.stack([x, y, occ], axis=1)
    return out.reshape(-1)

rgb_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

depth_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

class FusionDataset(Dataset):
    def __init__(self, df, dataset_root, rgb_transform=None, depth_transform=None):
        self.df = df.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.rgb_transform = rgb_transform
        self.depth_transform = depth_transform
        self.joint_cache = {}

    def __len__(self):
        return len(self.df)

    def _load_joint_file(self, joint_rel_path):
        if joint_rel_path not in self.joint_cache:
            mat = sio.loadmat(self.dataset_root / joint_rel_path)
            self.joint_cache[joint_rel_path] = mat["joints_gt"]
        return self.joint_cache[joint_rel_path]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rgb_img = Image.open(self.dataset_root / row["rgb_path"]).convert("RGB")
        if self.rgb_transform is not None:
            rgb_img = self.rgb_transform(rgb_img)

        depth_img = Image.open(self.dataset_root / row["depth_path"]).convert("L")
        if self.depth_transform is not None:
            depth_img = self.depth_transform(depth_img)

        joints_all = self._load_joint_file(row["joint_file"])
        frame = joints_all[:, :, int(row["frame_idx_0based"])]
        joint_vec = preprocess_joint_frame_xyo(frame)

        return {
            "rgb": rgb_img,
            "depth": depth_img,
            "joint": torch.tensor(joint_vec, dtype=torch.float32),
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "condition": row["condition"],
            "subject_id": row["subject_id"],
        }
    

# =========================
# Loader split
# =========================

def subject_wise_split(subject_ids, train_ratio=0.70, val_ratio=0.15, seed=42):
    subject_ids = sorted(subject_ids)
    rng = random.Random(seed)
    rng.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects


fusion_df = build_fusion_metadata(CSV_PATH)

subjects = sorted(fusion_df["subject_id"].unique().tolist())
train_subjects, val_subjects, test_subjects = subject_wise_split(subjects, 0.70, 0.15, 42)

train_df = fusion_df[fusion_df["subject_id"].isin(train_subjects)].reset_index(drop=True)
val_df   = fusion_df[fusion_df["subject_id"].isin(val_subjects)].reset_index(drop=True)
test_df  = fusion_df[fusion_df["subject_id"].isin(test_subjects)].reset_index(drop=True)

train_dataset = FusionDataset(train_df, DATASET_ROOT, rgb_transform, depth_transform)
val_dataset   = FusionDataset(val_df, DATASET_ROOT, rgb_transform, depth_transform)
test_dataset  = FusionDataset(test_df, DATASET_ROOT, rgb_transform, depth_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Encoder Model Definitions

Define the encoder architectures needed to load the pretrained weights.

At this stage, we only recreate the model structures required for:

- RGB encoder
- Depth encoder
- Joint encoder

These definitions should match the architectures used during the original encoder training notebooks.

In [63]:
# =========================
# Encoder feature dimensions
# =========================
DEPTH_FEATURE_DIM = 512
RGB_FEATURE_DIM = 128
JOINT_INPUT_DIM = 42
JOINT_FEATURE_DIM = 128

In [64]:
class DepthEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)

        # change first conv to 1 channel
        backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # remove FC
        layers = list(backbone.children())[:-1]

        self.encoder = nn.Sequential(*layers)
        self.flatten = nn.Flatten()

        self.feature_dim = 512

    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        return x


class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128)
        )
        self.feature_dim = 128

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection(x)
        return x


class JointEncoder(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, feature_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.feature_dim = feature_dim

    def forward(self, x):
        return self.encoder(x)

## Load Pretrained Encoders

Instantiate the three encoder models and load their encoder-only weights from the saved checkpoints.

After loading:
- move models to the selected device
- set them to evaluation mode
- freeze parameters so that only the gated fusion head is trained in this first experiment stage

In [65]:
# =========================
# Instantiate encoders
# =========================
depth_encoder = DepthEncoder().to(device)
rgb_encoder = RGBEncoder().to(device)
joint_encoder = JointEncoder(
    input_dim=JOINT_INPUT_DIM,
    hidden_dim=128,
    feature_dim=JOINT_FEATURE_DIM,
    dropout=0.3
).to(device)

# =========================
# Load checkpoints
# =========================
depth_ckpt = torch.load(DEPTH_CHECKPOINT_PATH, map_location=device)
rgb_ckpt = torch.load(RGB_CHECKPOINT_PATH, map_location=device)
joint_ckpt = torch.load(JOINT_CHECKPOINT_PATH, map_location=device)

depth_encoder.encoder.load_state_dict(depth_ckpt["encoder_state_dict"])
rgb_encoder.load_state_dict(rgb_ckpt["encoder_state_dict"])
joint_encoder.encoder.load_state_dict(joint_ckpt["encoder_state_dict"])

# =========================
# Freeze encoders
# =========================
for model in [depth_encoder, rgb_encoder, joint_encoder]:
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("All encoders loaded and frozen successfully.")

All encoders loaded and frozen successfully.


# Gated Fusion Model

In [66]:
# =========================
# Attention-based fusion head
# =========================
class AttentionFusionClassifier(nn.Module):
    def __init__(
        self,
        depth_encoder,
        rgb_encoder,
        joint_encoder,
        depth_feature_dim=512,
        rgb_feature_dim=128,
        joint_feature_dim=128,
        common_feature_dim=256,
        num_heads=4,
        ff_dim=512,
        dropout=0.1,
        num_classes=3,
    ):
        super().__init__()

        # frozen encoders
        self.depth_encoder = depth_encoder
        self.rgb_encoder = rgb_encoder
        self.joint_encoder = joint_encoder

        # project all modalities to same dim
        self.depth_proj = nn.Linear(depth_feature_dim, common_feature_dim)
        self.rgb_proj = nn.Linear(rgb_feature_dim, common_feature_dim)
        self.joint_proj = nn.Linear(joint_feature_dim, common_feature_dim)

        # modality-type embeddings (helps attention distinguish tokens)
        self.modality_embed = nn.Parameter(torch.randn(3, common_feature_dim))

        # one small transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=common_feature_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.attention_block = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(common_feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, depth_x, rgb_x, joint_x, return_features=False):
        with torch.no_grad():
            f_depth = self.depth_encoder(depth_x)   # [B, 512]
            f_rgb = self.rgb_encoder(rgb_x)         # [B, 128]
            f_joint = self.joint_encoder(joint_x)   # [B, 128]

        t_depth = self.depth_proj(f_depth)
        t_rgb = self.rgb_proj(f_rgb)
        t_joint = self.joint_proj(f_joint)

        # stack as 3 modality tokens
        tokens = torch.stack([t_rgb, t_depth, t_joint], dim=1)   # [B, 3, D]

        # add modality embeddings
        tokens = tokens + self.modality_embed.unsqueeze(0)

        # self-attention across modalities
        tokens = self.attention_block(tokens)  # [B, 3, D]

        # pool tokens
        fused = tokens.mean(dim=1)  # [B, D]

        logits = self.classifier(fused)

        if return_features:
            return logits, fused
        return logits
    
# =========================
# Creating fusion model
# =========================
fusion_model = AttentionFusionClassifier(
    depth_encoder=depth_encoder,
    rgb_encoder=rgb_encoder,
    joint_encoder=joint_encoder,
    depth_feature_dim=DEPTH_FEATURE_DIM,
    rgb_feature_dim=RGB_FEATURE_DIM,
    joint_feature_dim=JOINT_FEATURE_DIM,
    common_feature_dim=COMMON_FEATURE_DIM,
    num_heads=4,
    ff_dim=512,
    dropout=0.1,
    num_classes=NUM_CLASSES,
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, fusion_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

trainable_params = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in fusion_model.parameters() if not p.requires_grad)

print("Trainable params:", trainable_params)
print("Frozen params   :", frozen_params)

Trainable params: 758531
Frozen params   : 22532992


C:\Users\John joseph peter\AppData\Local\Temp\ipykernel_15596\2255670998.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.attention_block = nn.TransformerEncoder(encoder_layer, num_layers=1)


In [69]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score


def train_one_epoch_fusion(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        depth_x = batch["depth"].to(device)
        rgb_x = batch["rgb"].to(device)
        joint_x = batch["joint"].to(device)
        y = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(depth_x, rgb_x, joint_x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)

        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")

    return avg_loss, acc, f1

# =========================
# Evaluation Block
# =========================
@torch.no_grad()
def evaluate_fusion(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        depth_x = batch["depth"].to(device)
        rgb_x = batch["rgb"].to(device)
        joint_x = batch["joint"].to(device)
        y = batch["label"].to(device)

        logits = model(depth_x, rgb_x, joint_x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)

        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)

    return avg_loss, acc, f1, cm


# =========================
# Main Training block
# =========================
best_val_f1 = -1.0
best_ckpt_path = FUSION_CHECKPOINT_DIR / "best_attention_fusion.pth"

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch_fusion(
        fusion_model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc, val_f1, val_cm = evaluate_fusion(
        fusion_model, val_loader, criterion, device
    )

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"[Train] loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f}")
    print(f"[Val]   loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1

        torch.save(
            {
                "model_state_dict": fusion_model.state_dict(),
                "config": {
                    "common_feature_dim": COMMON_FEATURE_DIM,
                    "num_classes": NUM_CLASSES,
                    "num_heads": 4,
                    "ff_dim": 512,
                    "dropout": 0.1,
                },
                "best_val_f1": best_val_f1,
            },
            best_ckpt_path,
        )

        print(f"Saved best model -> {best_ckpt_path}")

# =========================
# Final evaluation on test set
# =========================
ckpt = torch.load(best_ckpt_path, map_location=device)
fusion_model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_acc, test_f1, test_cm = evaluate_fusion(
    fusion_model, test_loader, criterion, device
)

print("\n[Test]")
print(f"loss={test_loss:.4f} acc={test_acc:.4f} f1={test_f1:.4f}")
print("Confusion matrix:")
print(test_cm)


Epoch 1/15
[Train] loss=0.0176 acc=0.9957 f1=0.9957
[Val]   loss=0.0008 acc=0.9995 f1=0.9995
Saved best model -> C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\checkpoints_fusion_gated\best_attention_fusion.pth

Epoch 2/15
[Train] loss=0.0156 acc=0.9965 f1=0.9965
[Val]   loss=0.0001 acc=1.0000 f1=1.0000
Saved best model -> C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\checkpoints_fusion_gated\best_attention_fusion.pth

Epoch 3/15
[Train] loss=0.0137 acc=0.9975 f1=0.9975
[Val]   loss=0.0002 acc=1.0000 f1=1.0000

Epoch 4/15
[Train] loss=0.0137 acc=0.9974 f1=0.9974
[Val]   loss=0.0002 acc=1.0000 f1=1.0000

Epoch 5/15
[Train] loss=0.0140 acc=0.9971 f1=0.9971
[Val]   loss=0.0007 acc=1.0000 f1=1.0000

Epoch 6/15
[Train] loss=0.0112 acc=0.9976 f1=0.9976
[Val]   loss=0.0004 acc=1.0000 f1=1.0000

Epoch 7/15
[Train] loss=0.0103 acc=0.9973 f1=0.9973
[Val]   loss=0.0005 acc=1.0000 f1=1.0000

Epoch 8/15
[Train] loss=0.0094 acc=0.9978 f1=0.9978
[Val]